In [ ]:
####################################################################################
# SAAM Project 2026 - Part I
# CELL 4 - SECTION 2.2: OUT-OF-SAMPLE MVP RETURNS AND PERFORMANCE
####################################################################################

# This section computes the realised (out-of-sample) returns of the
# minimum variance portfolio. The weights are fixed at the end of year Y
# and applied to returns in year Y+1. We also allow weights to drift
# within the year based on realised returns.

# =============================================================================
# Load returns and weights
# =============================================================================

ri_returns = clean_headers(pd.read_csv(CLEAN_DIR / RI_RETURNS_FILE))
_, ret_isin_col = get_id_cols(ri_returns)
ret_month_cols = sort_month_cols(get_time_cols(ri_returns))
ri_ret_by_isin = ri_returns.set_index(ret_isin_col)

weights_by_year = clean_headers(pd.read_csv(OUT_DIR / MVP_W_FILE))

# =============================================================================
# Compute monthly out-of-sample returns
# =============================================================================

mvp_rows = []

for year in range(START_YEAR, END_YEAR + 1):

    investment_year = year + 1

    # Get weights for this portfolio
    year_weights = weights_by_year[
        weights_by_year["screen_year"] == year
    ].copy()

    if year_weights.empty:
        continue

    asset_list = year_weights["ISIN"].tolist()
    alpha = year_weights["weight"].to_numpy(dtype=float)

    # Get monthly columns for the investment year
    inv_cols = get_investment_year_cols(ret_month_cols, investment_year)

    if not inv_cols:
        continue

    # Extract returns (fill missing values with 0 for mid-year gaps)
    inv_returns = (
        ri_ret_by_isin.reindex(asset_list)[inv_cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0.0)
        .to_numpy(dtype=float)
    )

    # Loop through months
    for k, col in enumerate(inv_cols):

        rt = inv_returns[:, k]

        # Portfolio return
        rp = float(alpha @ rt)

        mvp_rows.append({
            "screen_year": year,
            "investment_year": investment_year,
            "date": pd.Timestamp(col),
            "Rp_mvp": rp,
        })

        # Update weights (drift)
        denom = 1.0 + rp

        if abs(denom) < 1e-12:
            # fallback: equal weights if something goes wrong
            alpha = np.ones_like(alpha) / len(alpha)
        else:
            alpha = alpha * (1.0 + rt) / denom
            alpha = np.clip(alpha, 0.0, None)

            # re-normalise
            s = alpha.sum()
            if s <= 1e-12:
                alpha = np.ones_like(alpha) / len(alpha)
            else:
                alpha = alpha / s

# Convert to dataframe
mvp_monthly_returns = (
    pd.DataFrame(mvp_rows)
    .sort_values("date")
    .reset_index(drop=True)
)

if mvp_monthly_returns.empty:
    raise RuntimeError("No MVP returns generated.")

# =============================================================================
# Compute cumulative returns
# =============================================================================

mvp_monthly_returns["cum_return"] = (
    (1.0 + mvp_monthly_returns["Rp_mvp"]).cumprod() - 1.0
)

# =============================================================================
# Compute performance statistics
# =============================================================================

mvp_stats = summary_stats(
    mvp_monthly_returns.set_index("date")["Rp_mvp"],
    "P(mv)_oos"
)

# =============================================================================
# Display results
# =============================================================================

print("\n" + "=" * 60)
print("Minimum Variance Portfolio (Out-of-Sample Performance)")
print("=" * 60)

display_stats = pd.DataFrame([mvp_stats])[
    ["portfolio", "ann_mean_return", "ann_volatility",
     "sharpe_ratio", "min_monthly_return", "max_monthly_return"]
].copy()

display_stats.columns = [
    "Portfolio", "Ann. Return", "Ann. Volatility",
    "Sharpe Ratio", "Min Monthly Ret.", "Max Monthly Ret."
]

for col in ["Ann. Return", "Ann. Volatility",
            "Min Monthly Ret.", "Max Monthly Ret."]:
    display_stats[col] = display_stats[col].apply(lambda x: f"{x:.2%}")

display_stats["Sharpe Ratio"] = display_stats["Sharpe Ratio"].apply(lambda x: f"{x:.3f}")

print(display_stats.to_string(index=False))

print("\nPeriod:",
      mvp_monthly_returns["date"].min().strftime("%b %Y"),
      "to",
      mvp_monthly_returns["date"].max().strftime("%b %Y"))

# =============================================================================
# Save outputs
# =============================================================================

mvp_monthly_returns.to_csv(OUT_DIR / MVP_RET_FILE, index=False)
pd.DataFrame([mvp_stats]).to_csv(OUT_DIR / "mvp_performance_summary.csv", index=False)

print("\nSection 2.2 complete.")